<!-- project-notebook-context -->
# Fall Event Detection CNN - Training Workflow Notebook

**Current project:** `fall-event-detection-cnn`

**Notebook role:** Notebook record for archive extraction, frame preprocessing, CNN training, evaluation, and video-level fall-event detection. The project is being expanded into a reproducible video event detection pipeline.

## Current project entry points

- `src/model.py`
- `scripts/model_summary.py`

## Maintenance notes

- Keep private videos and datasets outside Git.
- Future work should move preprocessing, training, and evaluation into scripts/ and src/.

---



In [1]:
import os, shutil, random, glob, zipfile
import numpy as np
import matplotlib.pyplot as plt
try:
    import cv2 as cv
except Exception:
    cv = None
try:
    import keras
    import tensorflow as tf
    from keras import models, optimizers
    from keras import backend as K
    from keras.models import Sequential, load_model
    from keras.layers import Dense, Activation, Dropout, Flatten, TimeDistributed, Conv2D, MaxPooling2D
    from keras.preprocessing.image import ImageDataGenerator
    from keras.utils import to_categorical
    from keras.optimizers import SGD, Adam
    from tensorflow.keras.utils import img_to_array
    from tensorflow.keras.preprocessing import image
    from sklearn.metrics import classification_report, confusion_matrix
    from sklearn.utils import shuffle
    HAVE_TF = True
except Exception as e:
    HAVE_TF = False
    print('TensorFlow/Keras stack not available:', e)

TensorFlow/Keras stack not available: No module named 'keras'


In [2]:
from pathlib import Path

base_dir = Path('../data/raw/training')
DATA_OK = base_dir.exists()
base_dir = str(base_dir)
RUN = HAVE_TF and DATA_OK
if not RUN:
    reasons = []
    if not HAVE_TF:
        reasons.append('TensorFlow/Keras not installed')
    if not DATA_OK:
        reasons.append(f'training data not found at {base_dir}')
    print('Pipeline cells are skipped: ' + '; '.join(reasons) + '.')
    print('This notebook is retained as a workflow record; the maintained code path lives in src/ and scripts/.')
else:
    print('Prerequisites satisfied; running full pipeline.')

Pipeline cells are skipped: TensorFlow/Keras not installed; training data not found at ..\data\raw\training.
This notebook is retained as a workflow record; the maintained code path lives in src/ and scripts/.


# Zip extract

In [3]:
if RUN:
    file_name0 = [f for f in os.listdir(base_dir+'/0') if os.path.isfile(os.path.join(base_dir+'/0',f))]

    file_name1 = [f for f in os.listdir(base_dir+'/1') if os.path.isfile(os.path.join(base_dir+'/1',f))]

    #print(file_name0[0],file_name1[0],len(file_name0),len(file_name1))

In [4]:
if RUN:
    def zip_extract(file_name,k):
        zf = zipfile.ZipFile(file_name, 'r')
        zf.extractall(k)

In [5]:
if RUN:
    for i in range(len(file_name0)):
        file_path = base_dir+'/0/'+file_name0[i]
        zip_extract(file_path,'0')

In [6]:
if RUN:
    for j in range(len(file_name1)):
        file_path = base_dir+'/1/'+file_name1[j]
        zip_extract(file_path,'1')

# File pre-processing

In [7]:
if RUN:
    def img_get(img_paths):

        global num1
        images = []

        for file in os.listdir(img_paths):
            #print(file)
            img = cv.imread(img_paths+'/'+file)
            img = cv.resize(img, (160,120), interpolation=cv.INTER_AREA)
            images.append(img)

        num1=len(images)

        return images

In [8]:
if RUN:
    def get_data(class_paths):

        train_data = []
        train_label = []

        val_data = []
        val_label =  []

        for index, paths in enumerate(class_paths):

            num = len(paths)
            eval_index = random.sample(paths,k=int(num*0.8))

            for path in paths:
                if path in eval_index:
                    img1 = img_get(path)
                    t_label = np.ones(num1) * index
                    train_data.extend(img1)
                    train_label.extend(t_label)
                else:
                    img2 = img_get(path)
                    v_label = np.ones(num1) * index
                    val_data.extend(img2)
                    val_label.extend(v_label)


        train_data = np.array(train_data, dtype=np.float32)/255
        train_label = np.array(train_label, dtype=np.int32)

        val_data = np.array(val_data, dtype=np.float32)/255
        val_label = np.array(val_label, dtype=np.int32)

        return train_data ,train_label ,val_data ,val_label

In [9]:
if RUN:
    daily = glob.glob(r'0/*')
    fall = glob.glob(r'1/*')

In [10]:
if RUN:
    train_image, train_label ,val_image, val_label = get_data([daily, fall])

In [11]:
if RUN:
    img_gen = ImageDataGenerator(featurewise_center=True,featurewise_std_normalization=True,horizontal_flip=True)

In [12]:
if RUN:
    img_gen.fit(train_image)
    img_gen.fit(val_image)

In [13]:
if RUN:
    x_train, y_train, x_test, y_test  = train_image, train_label ,val_image, val_label

In [14]:
if RUN:
    x_train, y_train = shuffle(x_train, y_train, random_state=46)

In [15]:
if RUN:
    print(x_train)

In [16]:
if RUN:
    print(y_train)

In [17]:
if RUN:
    path = 'y_train.txt'

    file = open (path, 'w')
    for num in range(len(y_train)):
        file.write(str(y_train[num]))

    file.close()

In [18]:
if RUN:
    path = 'val_label.txt'

    file = open (path, 'w')
    for num in range(len(val_label)):
        file.write(str(val_label[num]))

    file.close()

In [19]:
if RUN:
    print(x_train.shape,y_train.shape,x_test.shape,y_test.shape)

In [20]:
if RUN:
    np.savez('train_gray.npz', images=x_train, labels=y_train)

In [21]:
if RUN:
    np.savez('test_gray.npz', images=x_test, labels=y_test)

# CNN

In [22]:
if RUN:
    model = Sequential()

    model.add(Conv2D(32, kernel_size=(3, 3),activation='relu',padding='same',input_shape=(120,160,3)))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    model.add(Conv2D(128, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    model.add(Flatten())

    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.5))

    model.add(Dense(2, activation='softmax'))

In [23]:
if RUN:
    model.compile(loss = 'sparse_categorical_crossentropy',
                  optimizer = 'adam', 
                  metrics = ['accuracy'])

In [24]:
if RUN:
    model.summary()

In [25]:
if RUN:
    test = np.load('test_gray.npz')
    train = np.load('train_gray.npz')

    x_train, y_train  = train["images"], train["labels"]

    x_test, y_test  = test["images"], test["labels"]

In [26]:
if RUN:
    history = model.fit(x_train, y_train, validation_data = (x_test, y_test), batch_size=20, epochs=10)

In [27]:
if RUN:
    model.save('model.h5')

# Evaluation

In [28]:
if RUN:
    model = load_model('model.h5')

In [29]:
if RUN:
    model.summary()

# zip extract

In [30]:
if RUN:
    test_dir = r'Evaluation Dataset'

In [31]:
if RUN:
    file_name2 = [f for f in os.listdir(test_dir) if os.path.isfile(os.path.join(test_dir,f))]

In [32]:
if RUN:
    def zip_extract(file_name,k):
        zf = zipfile.ZipFile(file_name, 'r')
        zf.extractall(k)

In [33]:
if RUN:
    for i in range(len(file_name2)):
        file_path = test_dir+'/'+file_name2[i]
        zip_extract(file_path,'EvaluationDataset')

# file processing

In [34]:
if RUN:
    def img_get2(img_paths):
        images = []
        for file in os.listdir(img_paths):
            #print(img_paths)
            img = cv.imread(img_paths+'/'+file)
            img = cv.resize(img,(160, 120))
            images.append(img)

        return images

In [35]:
if RUN:
    def get_data2(class_paths):
        train = {}
        x=0

        for index, paths in enumerate(class_paths):
            #print(paths)
            for path in paths:
                class_images = []
                #print(path)
                img = img_get2(path)
                class_images.extend(img)
                #print(type(class_images))
                class_images = np.array(class_images).astype('float32')/255.0
                #print(type(class_images))

                train[x]=class_images
                x+=1

        return train 

In [36]:
if RUN:
    Evalu= glob.glob("EvaluationDataset/*")

In [37]:
if RUN:
    x_test = get_data2([Evalu])

In [38]:
#print(x_test[14],len(x_test),type(x_test))

In [39]:
if RUN:
    predicted_classes = {}

    for j in range(len(x_test)):
        predictions = model.predict(x_test[j])
        predicted_classes[j] = np.argmax(predictions, axis=1)

In [40]:
#print(len(predicted_classes),predicted_classes[5],len(predicted_classes[5]))

In [41]:
if RUN:
    filename = os.listdir('EvaluationDataset')

In [42]:
if RUN:
    path = 'output.txt'
    filename = os.listdir(test_dir)

    file = open (path, 'w')
    file.write('ImageName,label\n')
    for num in range(len(predicted_classes)):
        file.write(str(filename[num])+','+str(predicted_classes[num][0])+'\n')

    file.close()